In [1]:
#!/usr/bin/env python3
"""
Compute per-grid-cell humidity quantiles over the full time domain,
using chunking over lat/lon (recommended), compatible with old xarray + dask.
"""

import numpy as np
import xarray as xr

path = "/work/uc1275/u301827/02_MSE/full_midlatitude/era5_midlatitudes_JJA_compiled.nc"
out_path = "/work/uc1275/u301827/02_MSE/full_midlatitude/era5_midlatitudes_JJA_q_daily_quantiles.nc"

vars_to_do = ["q"]
qs = 1-np.array([0.10, 0.35, 0.5, 0.65, 0.85, 0.90, 0.95, 0.975, 0.99])

# Choose chunks (tune these)
chunks = {
    "time": -1,   # REQUIRED for old-xarray quantile over time
    "lat": 50,   # tune
    "lon": 200,   # tune
}

compress = True
complevel = 4


def main():
    ds0 = xr.open_dataset(path, chunks=chunks, cache=False)

    out_vars = {}
    for v in vars_to_do:
        if v not in ds0.data_vars:
            raise KeyError(f"Variable '{v}' not found. Available: {list(ds0.data_vars)}")

        da = ds0[v]

        # If your dataset uses different dimension names, adapt here:
        if "time" not in da.dims:
            print(f"Skipping '{v}' (no time dimension).")
            continue

        q_da = da.quantile(qs, dim="time", skipna=True)
        q_da = q_da.assign_coords(quantile=qs).rename(f"{v}_quantiles")
        out_vars[f"{v}_quantiles"] = q_da
        print(f"Prepared quantiles for '{v}'")

    ds_q = xr.Dataset(out_vars)
    ds_q.attrs.update(ds0.attrs)
    ds_q.attrs["quantiles_over"] = "time (entire domain)"
    ds_q.attrs["quantile_levels"] = ",".join(map(str, qs))

    encoding = {}
    if compress:
        for name in ds_q.data_vars:
            encoding[name] = {"zlib": True, "complevel": complevel}

    ds_q.to_netcdf(out_path, encoding=encoding)
    print(f"\nDone. Wrote quantiles dataset to:\n  {out_path}")


if __name__ == "__main__":
    main()


Prepared quantiles for 'q'

Done. Wrote quantiles dataset to:
  /work/uc1275/u301827/02_MSE/full_midlatitude/era5_midlatitudes_JJA_q_daily_quantiles.nc


In [2]:
#!/usr/bin/env python3
"""
Compute per-grid-cell quantile thresholds over the full time domain
and save to NetCDF (compatible with OLD xarray + dask).

Quantiles (fractions): 0.50, 0.75, 0.85, 0.90, 0.95, 0.975, 0.99

Key requirement (old xarray): when reducing over "time" with quantile(),
the "time" dimension must be ONE dask chunk. So we chunk time=-1 and
chunk spatial dims (lat/lon) for parallelism.
"""

import numpy as np
import xarray as xr

# -------------------------
# User settings
# -------------------------
path = "/work/uc1275/u301827/02_MSE/full_midlatitude/era5_midlatitudes_JJA_compiled.nc"
out_path = "/work/uc1275/u301827/02_MSE/full_midlatitude/era5_midlatitudes_JJA_t500_daily_quantiles.nc"

# Variable(s) to process
vars_to_do = ["t"]  # e.g., temperature variable name in your file

# Quantile levels (fractions)
qs = np.array([0.10, 0.35, 0.5, 0.65, 0.85, 0.90, 0.95, 0.975, 0.99])

# Chunking (tune lat/lon to your grid + memory)
# NOTE: time=-1 is REQUIRED for old-xarray quantile over time.
chunks = {
    "time": -1,
    "lat": 100,
    "lon": 200,
}

# Compression for output
compress = True
complevel = 4


def main():
    # -------------------------
    # Open lazily (dask-backed)
    # -------------------------
    ds0 = xr.open_dataset(path, chunks=chunks, cache=False)

    out_vars = {}

    for v in vars_to_do:
        if v not in ds0.data_vars:
            raise KeyError(
                f"Variable '{v}' not found in dataset.\n"
                f"Available variables: {list(ds0.data_vars)}"
            )

        da = ds0[v]

        # Skip anything without time
        if "time" not in da.dims:
            print(f"Skipping '{v}' (no 'time' dimension).")
            continue

        # OLD xarray-friendly quantile call (no method=..., no dask_gufunc_kwargs=...)
        q_da = da.quantile(
            qs,
            dim="time",
            skipna=True,
        )

        # Clean output
        q_da = q_da.assign_coords(quantile=qs).rename(f"{v}_quantiles")

        out_vars[f"{v}_quantiles"] = q_da
        print(f"Prepared quantiles for '{v}' -> '{v}_quantiles'")

    ds_q = xr.Dataset(out_vars)

    # Metadata
    ds_q.attrs.update(ds0.attrs)
    ds_q.attrs["quantiles_over"] = "time (entire domain)"
    ds_q.attrs["quantile_levels"] = ",".join(map(str, qs))

    # -------------------------
    # Write (triggers compute)
    # -------------------------
    encoding = {}
    if compress:
        for name in ds_q.data_vars:
            encoding[name] = {"zlib": True, "complevel": complevel}

    ds_q.to_netcdf(out_path, encoding=encoding)
    print(f"\nDone. Wrote quantiles dataset to:\n  {out_path}")


if __name__ == "__main__":
    main()


Prepared quantiles for 't' -> 't_quantiles'

Done. Wrote quantiles dataset to:
  /work/uc1275/u301827/02_MSE/full_midlatitude/era5_midlatitudes_JJA_t500_daily_quantiles.nc


In [5]:
1-np.array([0.10, 0.35, 0.5, 0.65, 0.85, 0.90, 0.95, 0.975, 0.99])

array([0.9  , 0.65 , 0.5  , 0.35 , 0.15 , 0.1  , 0.05 , 0.025, 0.01 ])